## Étape 1 — Chargement et fusion des données

In [ ]:
import pandas as pd

df = pd.read_csv("data/archelect_search.csv")
df.head()


In [ ]:
df.columns


In [ ]:
import numpy as np

NM = "non mentionné"

df["parti"] = np.select(
    [
        df["titulaire-liste"] != NM,
        df["suppleant-liste"]  != NM,
        df["titulaire-soutien"] != NM,
        df["suppleant-soutien"] != NM,
    ],
    [
        df["titulaire-liste"],
        df["suppleant-liste"],
        df["titulaire-soutien"],
        df["suppleant-soutien"],
    ],
    default=NM
)

print(df["parti"].value_counts().head(10))
print(f"\nNon mentionné : {(df['parti'] == NM).sum()} ({(df['parti'] == NM).mean()*100:.1f}%)")

In [ ]:
df.dtypes

In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d", errors="coerce")
df = df[df["date"].dt.year.isin([1973, 1978, 1981, 1988, 1993])]
df["date"].value_counts


In [ ]:
import os

TEXT_ROOT = "arkindex_archelec/text_files"

# Charger tous les textes dans un dictionnaire {id: texte}
texts = {}
for dirpath, _, files in os.walk(TEXT_ROOT):
    for fname in files:
        if fname.endswith(".txt"):
            doc_id = fname.replace(".txt", "")
            text = open(os.path.join(dirpath, fname), encoding="utf-8", errors="ignore").read()
            texts[doc_id] = text

# Joindre avec les métadonnées
df["text"] = df["id"].map(texts)

# Vérification
print(f"Textes chargés : {len(texts)}")
print(f"Lignes avec texte : {df['text'].notna().sum()}")
print(f"Lignes sans texte : {df['text'].isna().sum()}")


In [ ]:
# on vérifie que le texte est le bon
print(df["text"].iloc[0])

In [ ]:
# on voit que tous les textes chargés n'ont pas de ligne
print(*sorted(set(texts) - set(df["id"])), sep="\n")

# Ces fichiers (bulletins de vote sans doute), n'apporte pas d'info puisque on a le PF pour chqaue BV
# on peut donc les supprimer 


In [ ]:
# 27 lignes sans texte
df[df["text"].isna()][["id", "date", "contexte-election", "titulaire-nom", "titulaire-liste"]]

# pour les présidentielle noramal, on les a pas avant 93


# Étape 2 — Exploration du corpus ARCHELEC

In [ ]:
# on garde uniquement les lignes avec texte
df = df[df["text"].notna()].copy().reset_index(drop=True)
df["year"] = df["date"].dt.year



In [ ]:
#pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, ax = plt.subplots(figsize=(8, 5))

# --- Distribution par année ---
year_counts = df["year"].value_counts().sort_index()
ax.bar(year_counts.index.astype(str), year_counts.values, color="steelblue", edgecolor="white")
ax.set_title("Professions de foi par année d'élection")
ax.set_xlabel("Année")
ax.set_ylabel("Nombre de documents")
for bar, val in zip(ax.patches, year_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 str(val), ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()

## Distribution des partis/coallitions politiques

In [ ]:
TOP_N = 25

party_counts = (
    df["parti"]
    .replace("non mentionné", None)
    .dropna()
    .value_counts()
    .head(TOP_N)
)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(party_counts.index[::-1], party_counts.values[::-1], color="steelblue", edgecolor="white")
ax.set_title(f"Top {TOP_N} des partis/listes (professions de foi avec parti renseigné)")
ax.set_xlabel("Nombre de candidats")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
for bar, val in zip(bars, party_counts.values[::-1]):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            str(val), va="center", fontsize=9)
plt.tight_layout()
plt.show()

n_no_party = (df["parti"] == "non mentionné").sum()
print(f"Candidats sans parti renseigné : {n_no_party} ({n_no_party/len(df)*100:.1f}%)")

## Profil des candidats : sexe et âge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sexe_counts = (
    df["titulaire-sexe"]
    .replace("non mentionné", None)
    .dropna()
    .value_counts()
)
axes[0].pie(sexe_counts.values, labels=sexe_counts.index, autopct="%1.1f%%",
            colors=["steelblue", "coral", "brown"], startangle=90)
axes[0].set_title("Répartition par sexe (titulaire)")

# --- Tranche d'âge ---
age_order = [
    "moins de 30 ans", "entre 30 et 39 ans", "entre 40 et 49 ans",
    "entre 50 et 59 ans", "entre 60 et 69 ans", "70 ans et plus"
]
age_counts = (
    df["titulaire-age-tranche"]
    .replace("non mentionné", None)
    .dropna()
    .value_counts()
    .reindex(age_order)
    .dropna()
)
axes[1].bar(range(len(age_counts)), age_counts.values, color="mediumseagreen", edgecolor="white")
axes[1].set_xticks(range(len(age_counts)))
axes[1].set_xticklabels(age_counts.index, rotation=30, ha="right")
axes[1].set_title("Répartition par tranche d'âge (titulaire)")
axes[1].set_ylabel("Nombre de candidats")

plt.tight_layout()
plt.show()

## Évolution de la représentation féminine par élection

In [ ]:
df_sexe = df[df["titulaire-sexe"].isin(["homme", "femme"])]
femme_pct = (
    df_sexe.groupby("year")["titulaire-sexe"]
    .apply(lambda s: (s == "femme").mean() * 100)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(femme_pct.index, femme_pct.values, marker="o", color="coral", linewidth=2)
ax.set_title("Part des candidatures féminines par élection (%)")
ax.set_xlabel("Année")
ax.set_ylabel("% de femmes")
ax.set_xticks(femme_pct.index)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x)}"))
ax.set_ylim(0, femme_pct.max() * 1.3)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
for x, y in zip(femme_pct.index, femme_pct.values):
    ax.annotate(f"{y:.1f}%", (x, y), textcoords="offset points", xytext=(0, 8), ha="center")
plt.tight_layout()
plt.show()

## Top départements & professions

In [ ]:
prof_counts = (
    df["titulaire-profession"]
    .replace("non mentionné", None)
    .dropna()
    # certaines professions sont multiples, on prend la première
    .str.split(";").str[0].str.strip()
    .value_counts()
    .head(20)
)

# "professeur", "enseignant" et "enseignante" désignent la même catégorie 
# et apparaissent séparément  à regrouper si on veut une analyse plus propre, pareil pour docteur ....

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(prof_counts.index[::-1], prof_counts.values[::-1], color="mediumseagreen", edgecolor="white")
ax.set_title("Top 20 des professions des titulaires")
ax.set_xlabel("Nombre de candidats")

plt.tight_layout()
plt.show()

## Étape 2 - Prétraitement

In [ ]:
# Suppression du header "Sciences Po / fonds CEVIPOF" présent dans certains documents
df["text"] = df["text"].str.replace("Sciences Po / fonds CEVIPOF", "", regex=False)

In [ ]:
#pip install spacy scikit-learn

In [ ]:
#!python -m spacy download fr_core_news_sm


In [ ]:
import spacy
import os
import unicodedata

def strip_accents(text):
    return unicodedata.normalize('NFD', text).encode('ascii', 'ignore').decode('ascii')

STOPWORDS = [x.strip() for x in open("data/stop_word_fr.txt").readlines()]

SAVE_PATH = "data/lemmatized.csv"
BATCH_SIZE = 2000


# Lemmatisation (environ 15m)
# Si le fichier existe déjà, on charge ce qui a déjà été lemmatisé
if os.path.exists(SAVE_PATH):
    done = pd.read_csv(SAVE_PATH)
    done_ids = set(done["id"])
    print(f"{len(done)} textes déjà lemmatisés, {len(df) - len(done)} restants")
else:
    done = pd.DataFrame(columns=["id", "lemmatized_text"])
    done_ids = set()
    print("Aucune sauvegarde trouvée, on repart de zéro")

# On ne traite que les textes pas encore lemmatisés
remaining = df[~df["id"].isin(done_ids)][["id", "text"]].reset_index(drop=True)

if len(remaining) > 0:
    nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
    results = []

    for i, doc in enumerate(nlp.pipe(remaining["text"], batch_size=64)):
        results.append({
            "id": remaining.loc[i, "id"],
            "lemmatized_text": " ".join([strip_accents(token.lemma_) for token in doc])
        })
        # Sauvegarde tous les BATCH_SIZE textes
        if (i + 1) % BATCH_SIZE == 0:
            done = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
            done.to_csv(SAVE_PATH, index=False)
            results = []
            print(f"  {i + 1}/{len(remaining)} sauvegardés")

    # Sauvegarde du reste
    if results:
        done = pd.concat([done, pd.DataFrame(results)], ignore_index=True)
        done.to_csv(SAVE_PATH, index=False)
        print(f"  {len(remaining)}/{len(remaining)} sauvegardés")

# Jointure avec le dataframe principal
df = df.merge(done[["id", "lemmatized_text"]], on="id", how="left")

In [ ]:
print(df["lemmatized_text"].iloc[0])

In [ ]:
print(df["text"].iloc[0])

## Étape 3 — LDA (Latent Dirichlet Allocation)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import math


# token_pattern : uniquement les tokens alphabétiques (élimine ☐, chiffres, etc.)
TOKEN_PATTERN = r"(?u)\b[a-zA-ZÀ-ÿ][a-zA-ZÀ-ÿ]+\b"

n_features = 1000
n_topics = 10


def plot_top_words(model, vectorizer, n_top_words, title, n_cols=5):
    feature_names = vectorizer.get_feature_names_out()
    n_topics = len(model.components_)
    nb_lines = math.ceil(n_topics / n_cols)
    fig, axes = plt.subplots(nb_lines, n_cols, figsize=(30, nb_lines * 6), sharex=False)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[-n_top_words:]
        top_features = feature_names[top_features_ind]
        weights = topic[top_features_ind]
        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f"Topic {topic_idx + 1}", fontdict={"fontsize": 20})
        ax.tick_params(axis="both", which="major", labelsize=14)
        for spine in "top right left".split():
            ax.spines[spine].set_visible(False)
    # Masquer les axes vides si n_topics n'est pas un multiple de n_cols
    for idx in range(n_topics, len(axes)):
        axes[idx].set_visible(False)
    fig.suptitle(title, fontsize=30)
    plt.subplots_adjust(top=0.90, bottom=0.05, wspace=0.90, hspace=0.3)
    plt.show()

In [ ]:



tf_vectorizer = CountVectorizer(
    max_df=0.95,
    min_df=5,
    max_features=n_features,
    stop_words=STOPWORDS,
    token_pattern=TOKEN_PATTERN
)
tf = tf_vectorizer.fit_transform(df["lemmatized_text"].fillna(""))
print(f"Matrice DTM : {tf.shape[0]} docs × {tf.shape[1]} termes")

lda = LatentDirichletAllocation(
    n_components=n_topics,
    max_iter=10,
    learning_method="online",
    learning_offset=50.0,
    random_state=0,
)
lda.fit(tf)

plot_top_words(lda, tf_vectorizer, 10, "LDA Topics — ARCHELEC")

In [ ]:
#pip install pyLDAvis

In [ ]:
import pyLDAvis
import pyLDAvis.lda_model

pyLDAvis.enable_notebook()
pyLDAvis.lda_model.prepare(lda, tf, tf_vectorizer)

In [ ]:
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Vectorisation TF-IDF (NMF fonctionne mieux avec TF-IDF que CountVectorizer)
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=10,
    max_features=2000,
    stop_words=STOPWORDS,
    token_pattern=TOKEN_PATTERN
)
tfidf = tfidf_vectorizer.fit_transform(df["lemmatized_text"].fillna(""))
print(f"Matrice TF-IDF : {tfidf.shape[0]} docs × {tfidf.shape[1]} termes")

# Entraînement
n_topics = 10  # à ajuster après analyse de cohérence
nmf = NMF(n_components=n_topics, random_state=42)
W = nmf.fit_transform(tfidf)

feature_names = tfidf_vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(nmf.components_):
    top_words = [feature_names[i] for i in np.argsort(-topic)[:20]]
    print(f"Topic {topic_idx} : {', '.join(top_words)}")
    print()


In [ ]:
plot_top_words(nmf, tfidf_vectorizer, 15, "NMF Topics — ARCHELEC")


In [ ]:
# Normaliser W pour que la somme par ligne = 1
W_normalized = W / W.sum(axis=1, keepdims=True)

df["dominant_topic"] = np.argmax(W_normalized, axis=1)
df["topic_score"] = np.max(W_normalized, axis=1)

# Labelliser les topics
topic_label = {
    0:  "Politique sociale & économique",        # social, emploi, travail, société, droit
    1:  "Extrême droite / FN",                   # front, national, lepen, immigration, insécurité
    2:  "Gauche ouvrière / PCF-LO",              # travailleur, mitterrand, lutte, marchais, ouvrière
    3:  "Écologie radicale (bruit/Alsace?)",     # animaux, nature, marseille, bulletin → topic bruité
    4:  "Documents allemands (Alsace-Moselle)",  # die, und, der, fur → bruit linguistique
    5:  "Centre-droit / UDF — identité & sécu.", # udf, rpr, retablir, immigration, insécurité
    6:  "Écologie politique / Les Verts",         # écologie, écologiste, verts, planète, entente
    7:  "Candidature locale & mandat",            # maire, candidat, circonscription, dimanche, tour
    8:  "Gauche réformiste / PCF-PS",             # communiste, gauche, programme commun, changement
    9:  "Anti-patronat / Lutte des classes",      # patron, patronat, bourgeoisie, sacrifice, banque
}




In [ ]:
from gensim.corpora import Dictionary

analyzer = tfidf_vectorizer.build_analyzer()

tokenized_texts = [
    analyzer(doc)
    for doc in df["lemmatized_text"].fillna("")
]

dictionary = Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=10, no_above=0.95)
print(f"Dictionnaire : {len(dictionary)} termes")


In [ ]:
import matplotlib.pyplot as plt
import textwrap

topic_names = [f"{k}-{topic_label[k]}" for k in topic_label.keys()]

for row in df.sample(n=5, random_state=2026).itertuples():

    doc_id = row.Index
    topic_distribution = W_normalized[doc_id]
    dominant_topic = row.dominant_topic
    dominant_score = row.topic_score
    dominant_label = topic_label[dominant_topic]

    # Infos du candidat
    print(f"\nDocument {doc_id}")
    print(f"Candidat   : {row._15} {row._16}")         # titulaire-nom / prenom
    print(f"Parti      : {getattr(row, 'parti', '?')}")
    print(f"Année      : {row.year}")
    print(f"Topic dominant : {dominant_topic} — {dominant_label} ({dominant_score:.3f})\n")
    text = textwrap.fill(row.text, width=100)
    print(text[:1200])
    print("\n")

    # Distribution des topics
    plt.figure(figsize=(14, 4))
    bars = plt.bar(topic_names, topic_distribution)
    bars[dominant_topic].set_color("darkred")
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Proportion")
    plt.title(f"Distribution thématique — {row._15} {row._16} ({row.year})")
    plt.tight_layout()
    plt.show()


In [ ]:
#pip install bertopic sentence-transformers


In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer

docs = df["lemmatized_text"].fillna("").tolist()

# Modèle multilingue adapté au français
#embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#embeddings = embedding_model.encode(docs, show_progress_bar=True, batch_size=64)
#np.save("data/BERTopic/embeddings.npy", embeddings)

embeddings = np.load("data/BERTopic/embeddings.npy")



In [ ]:

vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    min_df=10,
    token_pattern=TOKEN_PATTERN
)

topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    nr_topics="auto",
    verbose=True
)

topics, probs = topic_model.fit_transform(docs, embeddings)
df["bertopic_topic"] = topics

topic_model.get_topic_info()


In [ ]:
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic.vectorizers import ClassTfidfTransformer


umap_model = UMAP(n_neighbors=10, n_components=5, min_dist=0.0, metric='cosine')

hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True  
)

vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,       
    token_pattern=TOKEN_PATTERN
)

ctfidf_model = ClassTfidfTransformer()
representation_model = KeyBERTInspired()

topic_model = BERTopic(
    umap_model=umap_model,
    embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2"),
    nr_topics="auto",
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    verbose=True
)





In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings)  
df["bertopic_topic"] = topics
topic_model.get_topic_info()

In [ ]:
import numpy as np

# Sauvegarder
# Ce que vous avez déjà
np.save("embeddings.npy", embeddings)
np.save("topics.npy", np.array(topics))
np.save("probs.npy", probs)
topic_model.save("bertopic_model")

# Distribution thématique par document (pour votre carte sémantique)
topic_distr, _ = topic_model.approximate_distribution(docs)
np.save("topic_distr.npy", topic_distr)

# Infos sur les topics (labels, taille, mots-clés)
topic_info = topic_model.get_topic_info()
topic_info.to_csv("topic_info.csv", index=False)

# Ajouter les topics au dataframe principal
df["bertopic_topic"] = topics
df.to_csv("df_with_topics.csv", index=False)




In [ ]:
topic_model.visualize_documents(
    docs=docs,
    embeddings=embeddings,
    hide_annotations=True,
    topics=[0, 1, 2, 3, 4, 5,6, 7, 8, 9],
    height=600,
    width=1000
)

In [ ]:
# --- 1. Carte intertopique (bulles proportionnelles à la taille du topic) ---
topic_model.visualize_topics()

In [ ]:
# --- 2. Carte des documents dans l'espace 2D ---
# pip install datamapplot
topic_model.visualize_document_datamap(
    docs,
    embeddings=embeddings,
    label_wrap_width=20,
    label_font_size=9,
)

In [ ]:
# --- 3. Dendrogramme hiérarchique des topics ---
hierarchical_topics = topic_model.hierarchical_topics(docs)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
# --- 4. Documents organisés hiérarchiquement ---
topic_model.visualize_hierarchical_documents(
    docs,
    hierarchical_topics,
    embeddings=embeddings,
    hide_annotations=True,
    height=800,
    width=1200,
)

In [ ]:
# --- 5. Heatmap de similarité entre topics ---
topic_model.visualize_heatmap()

In [ ]:
# --- 6. Distribution thématique d'un document ---
# probs de HDBSCAN = confidence du cluster assigné seulement (pas une distribution complète)
# approximate_distribution() calcule une vraie distribution sur tous les topics
topic_distr, _ = topic_model.approximate_distribution(docs[:1], window=4, stride=1)
topic_model.visualize_distribution(topic_distr[0])

## NB topic

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
from sklearn.decomposition import NMF
import numpy as np
import matplotlib.pyplot as plt

# --- Dictionnaire gensim à partir du même vectorizer ---
analyzer = tfidf_vectorizer.build_analyzer()
tokenized_texts = [analyzer(doc) for doc in df["lemmatized_text"].fillna("")]

dictionary = Dictionary(tokenized_texts)
dictionary.filter_extremes(no_below=10, no_above=0.95)

feature_names = tfidf_vectorizer.get_feature_names_out()

def get_topics(model, feature_names, n_top_words=10):
    topics = []
    for topic in model.components_:
        top_indices = np.argsort(-topic)[:n_top_words]
        topics.append([feature_names[i] for i in top_indices])
    return topics

# --- Calcul de cohérence pour plusieurs valeurs de k ---
topic_range = [5, 10, 15, 20, 25, 30]
coherence_metrics = ['u_mass', 'c_v', 'c_uci', 'c_npmi']
results = {met: {} for met in coherence_metrics}

for k in topic_range:
    nmf_k = NMF(n_components=k, random_state=42)
    nmf_k.fit_transform(tfidf)
    topics = get_topics(nmf_k, feature_names, n_top_words=10)

    for met in coherence_metrics:
        score = CoherenceModel(
            topics=topics,
            texts=tokenized_texts,
            dictionary=dictionary,
            coherence=met
        ).get_coherence()
        results[met][k] = score
        print(f"k={k:2d} | {met:8s} : {score:.4f}")

# --- Visualisation normalisée ---
plt.figure(figsize=(10, 6))
for met in coherence_metrics:
    ks = sorted(results[met].keys())
    scores = np.array([results[met][k] for k in ks])
    norm = (scores - scores.min()) / (scores.max() - scores.min() + 1e-9)
    plt.plot(ks, norm, marker='o', label=met)

plt.xlabel("Nombre de topics (k)")
plt.ylabel("Cohérence normalisée")
plt.title("Choix du nombre de topics — NMF ARCHELEC")
plt.xticks(topic_range)
plt.legend()
plt.grid(True)
plt.show()
